# Exploratory Analysis, Cleaning, And Feature Engineering

This notebook turns the raw LendingClub file into the canonical modeling table at `data/processed/loan_clean.csv`.
Each step is visible so the cleaning rules, temporal metadata handling, and feature engineering choices can be audited and adjusted manually.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in [Path.cwd(), Path.cwd().parent]
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook session.')

sys.path.append(str(PROJECT_ROOT / 'src'))

from IPython.display import display
import pandas as pd

from preprocess import (
    ADDITIONAL_CATEGORICAL_RAW_FEATURE_COLUMNS,
    ADDITIONAL_NUMERIC_RAW_FEATURE_COLUMNS,
    BASE_ENGINEERED_FEATURE_COLUMNS,
    BASE_RAW_FEATURE_COLUMNS,
    EXTENDED_ENGINEERED_FEATURE_COLUMNS,
    ISSUE_RAW_COLUMN,
    RAW_FEATURE_COLUMNS,
    STATUS_COLUMN,
    TARGET_COLUMN,
    add_engineered_features,
    clean_base_features,
    create_target,
    encode_categorical_features,
    load_raw_data,
    parse_issue_date_column,
    select_modeling_columns,
)

RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'accepted_2007_to_2018Q4.csv'
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'loan_clean.csv'
REQUIRED_RAW_COLUMNS = list(dict.fromkeys([STATUS_COLUMN, ISSUE_RAW_COLUMN] + RAW_FEATURE_COLUMNS))

print(f'Raw data path: {RAW_DATA_PATH}')
print(f'Processed data path: {PROCESSED_DATA_PATH}')


Raw data path: /Users/minleihao/Desktop/Risk Project/IDS583_Final_Project/data/raw/accepted_2007_to_2018Q4.csv
Processed data path: /Users/minleihao/Desktop/Risk Project/IDS583_Final_Project/data/processed/loan_clean.csv


## 1. Load Only The Columns Needed For Origination-Time Modeling

The raw LendingClub file contains many post-origination outcome fields that would leak future information.
This notebook loads only the issue-time fields used for target construction, cleaning, and candidate feature engineering.

In [2]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f'Raw dataset not found at {RAW_DATA_PATH}.')

raw_df = load_raw_data(RAW_DATA_PATH, usecols=REQUIRED_RAW_COLUMNS)
print(f'Loaded {len(raw_df):,} rows and {len(raw_df.columns):,} columns from raw data.')
display(raw_df.head())

Loaded 2,260,701 rows and 25 columns from raw data.


,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,...,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies
0,3600.0,36 months,13.99,123.03,C,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,...,1.0,7.0,0.0,2765.0,29.7,13.0,w,Individual,1.0,0.0
1,24700.0,36 months,11.99,820.28,C,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,...,4.0,22.0,0.0,21470.0,19.2,38.0,w,Individual,4.0,0.0
2,20000.0,60 months,10.78,432.66,B,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,...,0.0,6.0,0.0,7869.0,56.2,18.0,w,Joint App,5.0,0.0
3,35000.0,60 months,14.85,829.90,C,10+ years,MORTGAGE,110000.0,Source Verified,Dec-2015,...,0.0,13.0,0.0,7802.0,11.6,17.0,w,Individual,1.0,0.0
4,10400.0,60 months,22.45,289.91,F,3 years,MORTGAGE,104433.0,Source Verified,Dec-2015,...,3.0,12.0,0.0,21929.0,64.5,35.0,w,Individual,6.0,0.0


## 2. Construct The Target And Parse Temporal Metadata

The target is defined as `1` for `Charged Off` and `0` for `Fully Paid`.
At the same time, `issue_d` is parsed into `issue_date`, which is kept for temporal splitting only and never used as a predictive feature.

In [3]:
labeled_df = parse_issue_date_column(raw_df)
labeled_df = create_target(labeled_df)

status_summary = labeled_df[TARGET_COLUMN].value_counts(dropna=False).rename_axis('default').reset_index(name='rows')
print(f'Filtered to {len(labeled_df):,} target-valid rows.')
display(status_summary)
display(labeled_df[[ISSUE_RAW_COLUMN, 'issue_date', STATUS_COLUMN, TARGET_COLUMN]].head())

Filtered to 1,345,310 target-valid rows.


,default,rows
0,0,1076751
1,1,268559


,issue_d,issue_date,loan_status,default
0,Dec-2015,2015-12-01,Fully Paid,0
1,Dec-2015,2015-12-01,Fully Paid,0
2,Dec-2015,2015-12-01,Fully Paid,0
4,Dec-2015,2015-12-01,Fully Paid,0
5,Dec-2015,2015-12-01,Fully Paid,0


## 3. Keep The Modeling Columns

The baseline project started with a small set of borrower and loan descriptors.
This upgraded version keeps those variables and also retains additional origination-time numeric and low-cardinality categorical candidates so they can be screened later without re-reading the raw file.

In [4]:
modeling_df = select_modeling_columns(labeled_df)
column_groups = pd.DataFrame({
    'group': [
        'baseline raw columns',
        'additional numeric candidates',
        'additional categorical candidates',
    ],
    'columns': [
        ', '.join(BASE_RAW_FEATURE_COLUMNS),
        ', '.join(ADDITIONAL_NUMERIC_RAW_FEATURE_COLUMNS),
        ', '.join(ADDITIONAL_CATEGORICAL_RAW_FEATURE_COLUMNS),
    ],
})
display(column_groups)
display(modeling_df.head())

,group,columns
0,baseline raw columns,"loan_amnt, annual_inc, fico_range_low, dti, em..."
1,additional numeric candidates,"int_rate, installment, delinq_2yrs, inq_last_6..."
2,additional categorical candidates,"verification_status, grade, initial_list_statu..."


,sample_id,issue_date,loan_amnt,annual_inc,fico_range_low,dti,emp_length,home_ownership,term,purpose,...,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,verification_status,grade,initial_list_status,application_type,default
0,0,2015-12-01,3600.0,55000.0,675.0,5.91,10+ years,MORTGAGE,36 months,debt_consolidation,...,2765.0,29.7,13.0,1.0,0.0,Not Verified,C,w,Individual,0
1,1,2015-12-01,24700.0,65000.0,715.0,16.06,10+ years,MORTGAGE,36 months,small_business,...,21470.0,19.2,38.0,4.0,0.0,Not Verified,C,w,Individual,0
2,2,2015-12-01,20000.0,63000.0,695.0,10.78,10+ years,MORTGAGE,60 months,home_improvement,...,7869.0,56.2,18.0,5.0,0.0,Not Verified,B,w,Joint App,0
4,4,2015-12-01,10400.0,104433.0,695.0,25.37,3 years,MORTGAGE,60 months,major_purchase,...,21929.0,64.5,35.0,6.0,0.0,Source Verified,F,w,Individual,0
5,5,2015-12-01,11950.0,34000.0,690.0,10.20,4 years,RENT,36 months,debt_consolidation,...,8822.0,68.4,6.0,0.0,0.0,Source Verified,C,w,Individual,0


## 4. Clean Numeric Fields

This stage performs the deterministic data cleaning rules used by the project:
- parse `emp_length` into years
- extract `term_months` from `term`
- convert `int_rate` and `revol_util` from percentage strings into numeric values
- set invalid DTI, interest-rate, and utilization values to missing
- median-impute modeled numeric columns
- retain `issue_date` for later temporal validation

In [5]:
clean_df = clean_base_features(modeling_df)
clean_preview_columns = [
    'issue_date', 'loan_amnt', 'annual_inc', 'dti', 'emp_length', 'term_months',
    'int_rate', 'installment', 'revol_util', 'inq_last_6mths', 'pub_rec_bankruptcies',
    'missing_emp_length_flag',
]
display(clean_df[clean_preview_columns].head())
display(clean_df[clean_preview_columns].isna().mean().rename('missing_share').reset_index(name='missing_share'))

,issue_date,loan_amnt,annual_inc,dti,emp_length,term_months,int_rate,installment,revol_util,inq_last_6mths,pub_rec_bankruptcies,missing_emp_length_flag
0,2015-12-01,3600.0,55000.0,5.91,10.0,36.0,13.99,123.03,29.7,1.0,0.0,0
1,2015-12-01,24700.0,65000.0,16.06,10.0,36.0,11.99,820.28,19.2,4.0,0.0,0
2,2015-12-01,20000.0,63000.0,10.78,10.0,60.0,10.78,432.66,56.2,0.0,0.0,0
4,2015-12-01,10400.0,104433.0,25.37,3.0,60.0,22.45,289.91,64.5,3.0,0.0,0
5,2015-12-01,11950.0,34000.0,10.20,4.0,36.0,13.44,405.18,68.4,0.0,0.0,0


,index,missing_share
0,issue_date,0.0
1,loan_amnt,0.0
2,annual_inc,0.0
3,dti,0.0
4,emp_length,0.0
5,term_months,0.0
6,int_rate,0.0
7,installment,0.0
8,revol_util,0.0
9,inq_last_6mths,0.0


## 5. Create Feature Engineering Columns

The notebook keeps the original engineered features from the first refactor and adds several new origination-safe ratios and flags.
These new columns are intentionally transparent so they can be toggled on or off in the modeling notebook.

In [6]:
engineered_df = add_engineered_features(clean_df)
engineered_columns = pd.DataFrame({
    'group': ['base engineered', 'extended engineered'],
    'columns': [
        ', '.join(BASE_ENGINEERED_FEATURE_COLUMNS),
        ', '.join(EXTENDED_ENGINEERED_FEATURE_COLUMNS),
    ],
})
display(engineered_columns)
display(engineered_df[BASE_ENGINEERED_FEATURE_COLUMNS + EXTENDED_ENGINEERED_FEATURE_COLUMNS].head())

,group,columns
0,base engineered,"missing_emp_length_flag, log_annual_inc, loan_..."
1,extended engineered,"installment_to_income, revol_bal_to_income, hi..."


,missing_emp_length_flag,log_annual_inc,loan_to_income,high_dti_flag,long_term_flag,installment_to_income,revol_bal_to_income,high_revol_util_flag,recent_inquiry_flag,prior_delinquency_flag,bankruptcy_flag
0,0,10.915107,0.065455,0,0,0.002237,0.050273,0,0,0,0
1,0,11.082158,0.380000,0,0,0.012620,0.330308,0,1,1,0
2,0,11.050906,0.317460,0,1,0.006868,0.124905,0,0,0,0
4,0,11.556311,0.099585,0,1,0.002776,0.209982,0,1,1,0
5,0,10.434145,0.351471,0,0,0.011917,0.259471,0,0,0,0


## 6. One-Hot Encode Low-Cardinality Categoricals

Home ownership, purpose, FICO bucket, verification status, grade, initial list status, and application type are one-hot encoded with `drop_first=True`.
`issue_date` remains in the table for temporal splitting, but it is excluded later from the feature matrix.

In [7]:
encoded_df = encode_categorical_features(engineered_df)
print(f'Encoded modeling table shape: {encoded_df.shape[0]:,} rows x {encoded_df.shape[1]:,} columns')
display(encoded_df.head())

Encoded modeling table shape: 1,345,310 rows x 62 columns


,sample_id,issue_date,loan_amnt,annual_inc,fico_range_low,dti,emp_length,term_months,int_rate,installment,...,verification_status_Verified,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,initial_list_status_w,application_type_Joint App,default
0,0,2015-12-01,3600.0,55000.0,675.0,5.91,10.0,36.0,13.99,123.03,...,0,0,1,0,0,0,0,1,0,0
1,1,2015-12-01,24700.0,65000.0,715.0,16.06,10.0,36.0,11.99,820.28,...,0,0,1,0,0,0,0,1,0,0
2,2,2015-12-01,20000.0,63000.0,695.0,10.78,10.0,60.0,10.78,432.66,...,0,1,0,0,0,0,0,1,1,0
4,4,2015-12-01,10400.0,104433.0,695.0,25.37,3.0,60.0,22.45,289.91,...,0,0,0,0,0,1,0,1,0,0
5,5,2015-12-01,11950.0,34000.0,690.0,10.20,4.0,36.0,13.44,405.18,...,0,0,1,0,0,0,0,1,0,0


## 7. Save The Canonical Processed Dataset

The processed dataset written here is the single source of truth for the modeling notebook.
It includes `issue_date`, the cleaned base features, engineered features, categorical dummies, and `default`.

In [8]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
encoded_df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f'Saved {len(encoded_df):,} processed rows to {PROCESSED_DATA_PATH}.')
display(encoded_df.head())

Saved 1,345,310 processed rows to /Users/minleihao/Desktop/Risk Project/IDS583_Final_Project/data/processed/loan_clean.csv.


,sample_id,issue_date,loan_amnt,annual_inc,fico_range_low,dti,emp_length,term_months,int_rate,installment,...,verification_status_Verified,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,initial_list_status_w,application_type_Joint App,default
0,0,2015-12-01,3600.0,55000.0,675.0,5.91,10.0,36.0,13.99,123.03,...,0,0,1,0,0,0,0,1,0,0
1,1,2015-12-01,24700.0,65000.0,715.0,16.06,10.0,36.0,11.99,820.28,...,0,0,1,0,0,0,0,1,0,0
2,2,2015-12-01,20000.0,63000.0,695.0,10.78,10.0,60.0,10.78,432.66,...,0,1,0,0,0,0,0,1,1,0
4,4,2015-12-01,10400.0,104433.0,695.0,25.37,3.0,60.0,22.45,289.91,...,0,0,0,0,0,1,0,1,0,0
5,5,2015-12-01,11950.0,34000.0,690.0,10.20,4.0,36.0,13.44,405.18,...,0,0,1,0,0,0,0,1,0,0
